# Etapa 4. Simulação dos 50 municípios

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/fcoliveira-utfpr/milho_oeste/blob/main/notebooks/04_simulacao_oeste.ipynb)

Agora a simulação da Etapa 3 é repetida para **50 municípios × 27 anos (1999 a 2025) × 9 datas de semeadura**
(dias 5, 15 e 25 de janeiro, fevereiro e março): 12.150 ciclos. O resultado vai para o seu Google Drive e é a base
de todas as etapas seguintes.

Leva alguns minutos. Se o Colab desconectar no meio, rode de novo: os municípios já simulados ficam guardados e são
pulados.

In [ ]:
# ==========================================================================================
# CÉLULA 0. PREPARAR O AMBIENTE (rode sempre primeiro, toda vez que abrir o notebook)
# ==========================================================================================
# No Colab, esta célula:
#   1) instala as bibliotecas que o Colab não tem;
#   2) copia o projeto do GitHub (dados + funções) para a máquina do Colab;
#   3) conecta o seu Google Drive, onde os resultados ficam guardados entre uma sessão e outra.
# Fora do Colab (no computador do orientador), ela só acerta as pastas.
import os
import sys
from pathlib import Path

NO_COLAB = "google.colab" in sys.modules        # True quando o notebook roda no Colab

if NO_COLAB:
    !pip -q install agrometeorologiapy==1.0.1
    if Path("/content/milho_oeste").exists():
        !git -C /content/milho_oeste pull -q     # já copiado nesta sessão: só atualiza
    else:
        !git clone -q https://github.com/fcoliveira-utfpr/milho_oeste.git /content/milho_oeste
    os.chdir("/content/milho_oeste")             # entra na pasta do projeto
    from google.colab import drive
    drive.mount("/content/drive")                # pede autorização na primeira vez
    SAIDA = Path("/content/drive/MyDrive/milho_oeste/resultados")
else:
    if Path.cwd().name == "notebooks":
        os.chdir("..")
    SAIDA = Path("resultados")

SAIDA.mkdir(parents=True, exist_ok=True)         # cria a pasta de resultados, se não existir
sys.path.append("src")                           # permite importar as funções do projeto (pasta src/)
print("Pasta do projeto:", Path.cwd())
print("Resultados em   :", SAIDA)

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

import mza_oeste as M

clima = pd.read_parquet("dados/clima_oeste.parquet")
mun = pd.read_csv("dados/municipios_oeste.csv")

print("Anos:", M.ANOS[0], "a", M.ANOS[-1])
print("Datas de semeadura (mês, dia):", M.DATAS)
print("Total de ciclos:", len(mun) * len(M.ANOS) * len(M.DATAS))

## 1. O laço sobre os municípios

Para cada município (`itertuples` percorre as linhas da tabela), a função `M.simular_municipio` roda os 27 anos ×
9 datas e devolve uma tabela. Cada tabela é gravada num arquivo próprio, na pasta `simulacao_parcial`, para não
perder o trabalho se a sessão cair.

In [ ]:
import time

parcial = SAIDA / "simulacao_parcial"
parcial.mkdir(exist_ok=True)

inicio = time.time()
for k, m in enumerate(mun.itertuples(), start=1):
    arquivo = parcial / f"{m.cod_ibge}.parquet"
    if arquivo.exists():
        continue                                           # já simulado numa sessão anterior
    clima_m = clima[clima["cod_ibge"] == m.cod_ibge].set_index("data")
    tab = M.simular_municipio(clima_m, m.cod_ibge, m.latitude, m.dta_mm_m)
    tab.to_parquet(arquivo, index=False)
    print(f"{k:2d}/{len(mun)} {m.municipio:28s} {time.time() - inicio:5.0f} s")
print("Pronto.")

In [ ]:
# junta os 50 arquivos numa tabela só e grava
sim = pd.concat([pd.read_parquet(a) for a in sorted(parcial.glob("*.parquet"))], ignore_index=True)
sim = sim.merge(mun[["cod_ibge", "municipio"]], on="cod_ibge")
sim.to_parquet(SAIDA / "simulacao_mza_oeste.parquet", index=False)
print(sim.shape)
sim.head()

## 2. PP e PA por data de semeadura

A média dos 50 municípios e 27 anos para cada data. `pd.to_datetime(..., format="%d/%m")` transforma o texto
"05/02" em data, só para ordenar o eixo.

In [ ]:
por_data = sim.groupby("semeadura")[["ciclo_dias", "PPf", "PA"] + [f"ISNA_{f}" for f in M.FASES]].mean()
por_data = por_data.loc[sorted(por_data.index, key=lambda d: pd.to_datetime(d, format="%d/%m"))]

fig, ax = plt.subplots(figsize=(9, 4.5))
ax.plot(por_data.index, por_data["PPf"], marker="o", label="PP")
ax.plot(por_data.index, por_data["PA"], marker="s", label="PA (ky de Doorenbos & Kassam)")
ax.set_xlabel("Data de semeadura")
ax.set_ylabel("kg/ha")
ax.legend()
fig.savefig(SAIDA / "fig_pp_pa_data.png", dpi=300, bbox_inches="tight")
plt.show()
por_data.round(2)

## 3. A PA que se compara com o IBGE

O IBGE informa uma produtividade por município e ano, mas o produtor semeia ao longo de uma **janela**. A PA
comparável é a média das datas da janela usual: 15/01, 05/02, 25/02 e 15/03.

In [ ]:
jan = (sim[sim["semeadura"].isin(M.JANELA)]
       .groupby(["cod_ibge", "municipio", "ano"])[["PPf", "PA"]].mean().reset_index())
jan.groupby("municipio")[["PPf", "PA"]].mean().sort_values("PA").round(0)

### Exercício

1. Qual data de semeadura tem a maior PA média? E a maior PP? Por que não são a mesma? (olhe o ISNA de cada fase
   por data)
2. Quais os cinco municípios com a menor PA média? Eles têm algo em comum (CAD, localização)?

### O que escrever

No Material e Métodos, as datas de semeadura e o período simulado; nos Resultados, a variação da PP e da PA com a
data de semeadura (figura `fig_pp_pa_data.png`).

In [ ]:
# escreva sua resposta aqui